## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [2]:
# Directory paths

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Paloma/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"

downloads_saved = home + "downloads/" 

complete_files = home + "complete/" 
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")


In [11]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences_swine_y2k.csv")
metadata["Accession"].apply(lambda x: x.split(".")[0])
print(len(metadata))

paloma_fastas = []
for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file).split("/")[-1]
        if " new sequences 310125" in file_name:
            fasta_df = df_from_fasta(file_name)
            fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split(" ")[0].replace(">", "").split(".")[0])
            paloma_fastas.append(fasta_df)

# paloma_metadata_2019 = pd.read_csv("paloma_isolates_metadata_2019.csv")
# paloma_metadata_2019 = paloma_metadata_2019.rename(columns={"Strain name":"strain_name"})
# paloma_metadata_2016 = pd.read_csv("paloma_isolates_metadata_2016.csv")
# paloma_metadata_2016 = paloma_metadata_2016.rename(columns={"STRAIN NAME":"strain_name"})

# paloma_isolates_metadata = pd.concat([paloma_metadata_2016, paloma_metadata_2019]).drop_duplicates(subset="strain_name", keep="first")
# paloma_isolates_metadata["Isolate"] = paloma_isolates_metadata["strain_name"].apply(lambda x: x.split("/")[-2] if "Catalonia" not in x and "Navarra" not in x else x.split("/")[2])

# paloma_isolates_metadata 

paloma_fastas_metadata = []
for fasta in paloma_fastas:
    fasta_metadata = fasta.merge(metadata, on="Accession") 
    paloma_fastas_metadata.append(fasta_metadata)   

paloma_fastas_metadata

29199


[Empty DataFrame
 Columns: [full_header, sequence, Accession, GenBank_RefSeq, Assembly, SRA_Accession, BioSample, BioProject, Organism_Name, Species, Genus, Family, Genotype, Isolate, Segment, GenBank_Title, Length, Nuc_Completeness, Geo_Location, Country, USA, Host, Tissue_Specimen_Source, Submitters, Organization, Org_location, Publications, Collection_Date, Release_Date, Molecule_type]
 Index: []
 
 [0 rows x 30 columns],
 Empty DataFrame
 Columns: [full_header, sequence, Accession, GenBank_RefSeq, Assembly, SRA_Accession, BioSample, BioProject, Organism_Name, Species, Genus, Family, Genotype, Isolate, Segment, GenBank_Title, Length, Nuc_Completeness, Geo_Location, Country, USA, Host, Tissue_Specimen_Source, Submitters, Organization, Org_location, Publications, Collection_Date, Release_Date, Molecule_type]
 Index: []
 
 [0 rows x 30 columns],
 Empty DataFrame
 Columns: [full_header, sequence, Accession, GenBank_RefSeq, Assembly, SRA_Accession, BioSample, BioProject, Organism_Name, S

## Add sequences to dataframe

In [4]:
# NCBI Virus Naming Convention:
# "Accession|GenBank_Title|Assembly|SRA Accession|BioSample|BioProject|Genotype|Isolate|Geo Location|Host|Collection Date"

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences_swine_y2k.fasta") # Turn fasta into a dataframe

# sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))

# print(sequences_fasta["full_header"])

print((sequences_fasta)) 
# print(sequences_fasta.head())
print((metadata))

# Add sequences to the dataframe
metadata_segments = pd.concat([metadata, sequences_fasta], axis=1) # , "Segment"])

                                             full_header  \
0      >MK303327.1 |Influenza A virus (A/swine/France...   
1      >MK303328.1 |Influenza A virus (A/swine/France...   
2      >MK303329.1 |Influenza A virus (A/swine/France...   
3      >MK303330.1 |Influenza A virus (A/swine/France...   
4      >MK303331.1 |Influenza A virus (A/swine/France...   
...                                                  ...   
29194  >DQ118160.1 |Influenza A virus (A/swine/Potsda...   
29195  >DQ118161.1 |Influenza A virus (A/swine/Schwer...   
29196  >DQ118162.1 |Influenza A virus (A/swine/Schwer...   
29197  >DQ118163.1 |Influenza A virus (A/swine/Potsda...   
29198  >DQ118164.1 |Influenza A virus (A/swine/Potsda...   

                                                sequence  
0      AGCGAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAGC...  
1      ATGGATGTCAATCCGACTCTACTTTTCCTAAAAATTCCAGCACAAA...  
2      ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...  
3      ATGAAGGCAATACTAGTAGTTCTGCTGTATACATTT

In [5]:
paloma_sequences = paloma_isolates_metadata.merge(metadata_segments, on="Isolate", how="left") # .drop_duplicates(subset="strain_name", keep="first")

print(paloma_sequences)

paloma_sequences_genbank = paloma_sequences.dropna(subset="sequence")

print(paloma_sequences_genbank)

                    strain_name Collection date Province      Breed Subtype  \
0    A/swine/Spain/45700-1/2016      11/18/2016   Toledo  White pig     NaN   
1    A/swine/Spain/40250-2/2016      11/18/2016   Toledo  White pig     NaN   
2    A/swine/Spain/40192-1/2016      12/20/2016  Segovia  White pig     NaN   
3    A/swine/Spain/40250-1/2016      12/20/2016  Segovia  White pig     NaN   
4    A/swine/Spain/45690-1/2016      12/27/2016   Toledo  White pig     NaN   
..                          ...             ...      ...        ...     ...   
423         A/Navarra/4050/2022      10/17/2022  Navarra      human    H1N1   
424         A/Navarra/4050/2022      10/17/2022  Navarra      human    H1N1   
425         A/Navarra/4050/2022      10/17/2022  Navarra      human    H1N1   
426         A/Navarra/4050/2022      10/17/2022  Navarra      human    H1N1   
427         A/Navarra/4050/2022      10/17/2022  Navarra      human    H1N1   

     Isolate   Accession GenBank_RefSeq         Ass

In [6]:
print(paloma_sequences_genbank.columns)

# metadata_segments = metadata_segments.dropna(subset=["Isolate", "Collection_Date", "GenBank_Title", "Geo_Location", "Segment"])

# print(metadata_segments["Isolate"])

Index(['strain_name', 'Collection date', 'Province', 'Breed', 'Subtype',
       'Isolate', 'Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession',
       'BioSample', 'BioProject', 'Organism_Name', 'Species', 'Genus',
       'Family', 'Genotype', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'full_header', 'sequence'],
      dtype='object')


In [7]:
# Cut down to only columns we want
metadata_genoflu = paloma_sequences_genbank[["Accession", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Assembly", "Isolate", "Genotype", "Geo_Location", "full_header", "sequence", "Segment", "strain_name", "Breed", "Province"]] # .iloc[:, 1:]

# # Get genbank strain name
metadata_genoflu["genbank_isolate"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1].split("/")[-2])
# metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1])

metadata_genoflu # [metadata_genoflu["Genotype"]  == "B3.13"]

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Assembly,Isolate,Genotype,Geo_Location,full_header,sequence,Segment,strain_name,Breed,Province,genbank_isolate
23,PQ047761.1,Influenza A virus (A/swine/Spain/6370-1/2018(H...,Sus scrofa,2018-02-06,NaN,GCA_040888365.1,6370-1,H1N2,Spain,>PQ047761.1 |Influenza A virus (A/swine/Spain/...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAAC...,1.0,A/swine/Spain/6370-1/2018,Iberian pig,Badajoz,6370-1
24,PQ047762.1,Influenza A virus (A/swine/Spain/6370-1/2018(H...,Sus scrofa,2018-02-06,NaN,GCA_040888365.1,6370-1,H1N2,Spain,>PQ047762.1 |Influenza A virus (A/swine/Spain/...,AGCAAAAGCAGGCAAACTATTTGAATGGATGTCAATCCGACTCTAC...,2.0,A/swine/Spain/6370-1/2018,Iberian pig,Badajoz,6370-1
25,PQ047763.1,Influenza A virus (A/swine/Spain/6370-1/2018(H...,Sus scrofa,2018-02-06,NaN,GCA_040888365.1,6370-1,H1N2,Spain,>PQ047763.1 |Influenza A virus (A/swine/Spain/...,AGCAAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAAT...,3.0,A/swine/Spain/6370-1/2018,Iberian pig,Badajoz,6370-1
26,PQ047764.1,Influenza A virus (A/swine/Spain/6370-1/2018(H...,Sus scrofa,2018-02-06,NaN,GCA_040888365.1,6370-1,H1N2,Spain,>PQ047764.1 |Influenza A virus (A/swine/Spain/...,AGCAAAAGCAGGGGATAATTAAATCAACCAAAATGGAGGCAAAACT...,4.0,A/swine/Spain/6370-1/2018,Iberian pig,Badajoz,6370-1
27,PQ047765.1,Influenza A virus (A/swine/Spain/6370-1/2018(H...,Sus scrofa,2018-02-06,NaN,GCA_040888365.1,6370-1,H1N2,Spain,>PQ047765.1 |Influenza A virus (A/swine/Spain/...,AGCAAAAGCAGGGTAGATAATCACTCAATGAGTGACATCGAAGCCA...,5.0,A/swine/Spain/6370-1/2018,Iberian pig,Badajoz,6370-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,MK361760.1,Influenza A virus (A/swine/Osnabrueck/4050/200...,Sus scrofa,2005-04-19,NaN,GCA_038257315.1,4050,H3N2,Germany: Osnabrueck,>MK361760.1 |Influenza A virus (A/swine/Osnabr...,AGCAAAAGCAGGGGATAATTTTATCAACTATGAAGACTGTCATTGC...,4.0,A/Navarra/4050/2022,human,Navarra,4050
424,MK361761.1,Influenza A virus (A/swine/Osnabrueck/4050/200...,Sus scrofa,2005-04-19,NaN,GCA_038257315.1,4050,H3N2,Germany: Osnabrueck,>MK361761.1 |Influenza A virus (A/swine/Osnabr...,AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATTCACACCA...,5.0,A/Navarra/4050/2022,human,Navarra,4050
425,MK361762.1,Influenza A virus (A/swine/Osnabrueck/4050/200...,Sus scrofa,2005-04-19,NaN,GCA_038257315.1,4050,H3N2,Germany: Osnabrueck,>MK361762.1 |Influenza A virus (A/swine/Osnabr...,AGCAAAAGCAGGAGTTAAAATGAATCCAAATCAAAAGATAATAACA...,6.0,A/Navarra/4050/2022,human,Navarra,4050
426,MK361763.1,Influenza A virus (A/swine/Osnabrueck/4050/200...,Sus scrofa,2005-04-19,NaN,GCA_038257315.1,4050,H3N2,Germany: Osnabrueck,>MK361763.1 |Influenza A virus (A/swine/Osnabr...,AGCAAAAGCAGGTAGATATTTAAAGATGAGTCTTCTAACCGAGGTC...,7.0,A/Navarra/4050/2022,human,Navarra,4050


## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [8]:
# paloma_sequences_genbank = metadata_genoflu


# metadata_genoflu["full_header"] = metadata_genoflu["full_header"].apply(lambda x: x.replace("other_mammal", "Iberian") if x.split("/")[3] in iberian_swine else x.replace("other_mammal", "swine") if "sw" in x.lower() or "scrofa" in x.lower() or "boar" in x.lower() else x)

# metadata_genoflu["full_header"]


In [9]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Add more animal information from paloma metadata
print(paloma_isolates_metadata.columns)
# For each genbank_name, check if in paloma isolates metadata and rename host type to be pig breed. Else stay the same.

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y") if x == x else x)

# Get geographic locations
metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"]
# For each genbank name, check if in paloma isolates metadata and rename geo_location to include province. Else stay the same.



metadata_genoflu["Host_Type_Breed"] = metadata_genoflu["Breed"].apply(lambda x: x.split(" ")[0] if x == x else x)
# If Breed == NaN, fill with host_type
metadata_genoflu["Host_Type"] = metadata_genoflu["Host_Type_Breed"].fillna(metadata_genoflu["Host_Type"].apply(lambda x: x.replace("other_mammal", "swine") if "sw" in x.lower() or "scrofa" in x.lower() or "boar" in x.lower() else x))

metadata_genoflu["Geo_Location_Abrv_Province"] = metadata_genoflu["Geo_Location_Abrv"] + "-" + metadata_genoflu["Province"]
metadata_genoflu["Geo_Location_Abrv_Province"] = metadata_genoflu["Geo_Location_Abrv_Province"].fillna(metadata_genoflu["Geo_Location_Abrv"])


print(metadata_genoflu.columns)

# print(metadata_genoflu.dropna(subset="Accession"))

Index(['strain_name', 'Collection date', 'Province', 'Breed', 'Subtype',
       'Isolate'],
      dtype='object')
Index(['Accession', 'GenBank_Title', 'Host', 'Collection_Date',
       'SRA_Accession', 'Assembly', 'Isolate', 'Genotype', 'Geo_Location',
       'full_header', 'sequence', 'Segment', 'strain_name', 'Breed',
       'Province', 'genbank_isolate', 'Host_Type', 'Years',
       'Geo_Location_Abrv', 'Host_Type_Breed', 'Geo_Location_Abrv_Province'],
      dtype='object')


In [10]:
# If there is no SRA Accession, replace identifier with Assembly
# metadata_genoflu['SRA_Accession'] = np.where(metadata_genoflu['SRA_Accession'] == "", metadata_genoflu['Accession'].apply(lambda x: x.split(".")[0]), metadata_genoflu['SRA_Accession'])

# Make new labels
names = ">" + metadata_genoflu["Accession"] + "|" + metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1]) + "|" + metadata_genoflu["Genotype"] + "|" + metadata_genoflu["Geo_Location_Abrv_Province"] + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] 

metadata_genoflu["full_header"] = names
# metadata_genoflu["Name"] = metadata_genoflu["Name"].apply(lambda x: x.replace("other_mammal", "Iberian_swine") if x.split("/")[3] in iberian_swine else x.replace("other_mammal", "swine") if "sw" in x.lower() or "scrofa" in x.lower() or "boar" in x.lower() else x)

metadata_genoflu = metadata_genoflu[metadata_genoflu["Host_Type_Breed"] != "human"] # Get rid of human sequences, those are for GISAID

print(metadata_genoflu)

      Accession                                      GenBank_Title  \
23   PQ047761.1  Influenza A virus (A/swine/Spain/6370-1/2018(H...   
24   PQ047762.1  Influenza A virus (A/swine/Spain/6370-1/2018(H...   
25   PQ047763.1  Influenza A virus (A/swine/Spain/6370-1/2018(H...   
26   PQ047764.1  Influenza A virus (A/swine/Spain/6370-1/2018(H...   
27   PQ047765.1  Influenza A virus (A/swine/Spain/6370-1/2018(H...   
..          ...                                                ...   
414  PQ107779.1  Influenza A virus (A/swine/Spain/50700-1/2022(...   
415  PQ107780.1  Influenza A virus (A/swine/Spain/50700-1/2022(...   
416  PQ107781.1  Influenza A virus (A/swine/Spain/50700-1/2022(...   
417  PQ107782.1  Influenza A virus (A/swine/Spain/50700-1/2022(...   
418  PQ107783.1  Influenza A virus (A/swine/Spain/50700-1/2022(...   

           Host Collection_Date SRA_Accession         Assembly  Isolate  \
23   sus scrofa      2018-02-06                GCA_040888365.1   6370-1   
24   sus 

## Rename segments and make complete FASTA files

In [11]:
# Set up segments
serotypes = ["H1", "H3", "N1", "N2"]
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for serotype in serotypes:
    # metadata_genoflu["Serotype"] = metadata_genoflu["Name"].apply(lambda x: x.split("|")[2])
    m_g_s = metadata_genoflu[metadata_genoflu["Genotype"].str.contains(serotype)]
    m_g_s["Serotype"] = serotype
    for segment in segments.values():
        m_g = m_g_s[m_g_s["Segment_Name"] == segment]
        print(m_g)
        segment_genotype_dfs.append(m_g)
    # for genotype in list(set(m_g["Genotype"].values)):
    #     if genotype in genotypes:
    #         df = m_g[(m_g["Genotype"] == genotype)] 
    #         # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
    #         segment_genotype_dfs.append(df)
    #         pair = genotype + "_" + segment
    #         print(pair)

      Accession                                      GenBank_Title  \
23   PQ047761.1  Influenza A virus (A/swine/Spain/6370-1/2018(H...   
69   PQ107519.1  Influenza A virus (A/swine/Spain/6370-8/2019(H...   
77   PQ107529.1  Influenza A virus (A/swine/Spain/6370-9/2019(H...   
85   PQ107537.1  Influenza A virus (A/swine/Spain/6370-10/2019(...   
93   PQ107554.1  Influenza A virus (A/swine/Spain/6370-11/2019(...   
108  PP338874.1  Influenza A virus (A/swine/Spain/22269-1/2020(...   
115  PP330401.1  Influenza A virus (A/swine/Spain/31310-1/2020(...   
123  PP330414.1  Influenza A virus (A/swine/Spain/22070-1/2020(...   
130  PP330721.1  Influenza A virus (A/swine/Spain/50070-1/2020(...   
138  PP330730.1  Influenza A virus (A/swine/Spain/32070-1/2020(...   
146  PP330763.1  Influenza A virus (A/swine/Spain/22270-1/2020(...   
154  PP331797.1  Influenza A virus (A/swine/Spain/40217-1/2020(...   
162  PP331788.1  Influenza A virus (A/swine/Spain/05165-1/2020(...   
170  PP335236.1  Inf

In [12]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    # print(df)
    if len(df) > 0:
        for serotype in serotypes:
            if df["Serotype"].values[0] == serotype:
                print(df)
                df.to_csv("GenBank_" + df["Segment_Name"].values[0] + "_" + serotype + "_metadata_Paloma.csv")
                df = df.reset_index()
                if len(df) > 0: # If there are entries
                    file_name = "GenBank_" + df["Segment_Name"].values[0] + "_" + serotype + "_Paloma.fasta"
                    output_file = open(complete_files + file_name, "w")

                    for index, row in df.iterrows():
                        name = df.loc[index, "full_header"]
                        name = name.replace(" ", "_")
                        # names.append(name)
                        sequence = df.loc[index, "sequence"]
                        # First is header, second is sequence
                        output_file.write(name + "\n")
                        output_file.write(sequence + "\n")
                    
                output_file.close()

      Accession                                      GenBank_Title  \
23   PQ047761.1  Influenza A virus (A/swine/Spain/6370-1/2018(H...   
69   PQ107519.1  Influenza A virus (A/swine/Spain/6370-8/2019(H...   
77   PQ107529.1  Influenza A virus (A/swine/Spain/6370-9/2019(H...   
85   PQ107537.1  Influenza A virus (A/swine/Spain/6370-10/2019(...   
93   PQ107554.1  Influenza A virus (A/swine/Spain/6370-11/2019(...   
108  PP338874.1  Influenza A virus (A/swine/Spain/22269-1/2020(...   
115  PP330401.1  Influenza A virus (A/swine/Spain/31310-1/2020(...   
123  PP330414.1  Influenza A virus (A/swine/Spain/22070-1/2020(...   
130  PP330721.1  Influenza A virus (A/swine/Spain/50070-1/2020(...   
138  PP330730.1  Influenza A virus (A/swine/Spain/32070-1/2020(...   
146  PP330763.1  Influenza A virus (A/swine/Spain/22270-1/2020(...   
154  PP331797.1  Influenza A virus (A/swine/Spain/40217-1/2020(...   
162  PP331788.1  Influenza A virus (A/swine/Spain/05165-1/2020(...   
170  PP335236.1  Inf

In [13]:
metadata_genoflu["full_header"].values[0]

paloma_sequences_genbank = metadata_genoflu

## GISAID

In [14]:
# Function to get metadata
def separate_fasta_by_segs(metadata, fasta, animals_df, genotypes): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first

    unique_segments = list(set(fasta["Segment"])) # Get list of segments

    # “>EPI_ID|Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for genotype in genotypes: # .keys(): # For each genotype
        print(genotype)
        for seg in unique_segments: # For each segment

            xls = metadata # [metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == genotype] # Get only the metadata corresponding to that genotype
            xls = xls.rename(columns={"Isolate_Id":"Identifier"})
            print(xls)
            # print("XLS: ", metadata["Genotype"])
            # print(xls["Clade"])

            # print(d11_xls)

            # FASTA
            # if "Identifier" in fasta.columns:
            #     mask = fasta["Identifier"].isin(xls['Isolate_Id'])
            # else:           
            #     mask = fasta['Isolate_Id'].isin(xls['Isolate_Id'])

            # fasta_seg_pre = fasta[fasta["Identifier"].isin(xls['Isolate_Id'])] # Get only the identifiers (Isolate_Id) that are left after metadata is filtered for genotype
            fasta_seg_pre = fasta.merge(xls, how="inner", on="Identifier")
            print(fasta_seg_pre.columns)
            # break 

            # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
            fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

            fasta_seg["Genotype"] = genotype

            # Rename sequences 
            new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name_x"] + "|" + fasta_seg["Subtype_x"] + "|" + fasta_seg["Location"].apply(lambda x: x.split(" / ")[1] if x.split(" / ")[0] == "Europe" else x.split(" / ")[0]) + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) 
            fasta_seg["full_header"] = new_name
            # print(fasta_seg["New_Name"])

            segment_fastas.append(fasta_seg)
            print(fasta_seg)

            segment_fastas.append(fasta_seg)

    return segment_fastas, unique_segments

def fasta_df(file_name):
    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    segments = []
    collection_dates = []
    sequences = []
    # host_types = []
    species = []
    identifiers = []
    locations = []
    # genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    try:
                        split_header = header.split("|")
                        if len(header.split("|")) > 4:
                            identifier = header.split("|")[0]
                            identifiers.append(identifier)
                            split_first_header = split_header[1].split("/")
                        # else:
                        #     identifiers.append("unknown")
                        #     split_first_header = split_header[0].split("/")
                        # print(split_first_header)
                        # print(split_header)
                        headers.append(header) 
                        isolate_ids.append(split_first_header[-2])
                        locations.append(split_first_header[-3])
                        isolate_names.append(split_header[-4]) # We'll need to extract data from this too
                        # print(split_header[2].split("_")[-1])
                        subtypes.append(split_header[-3].split("_")[-1])  # Get only H5N1
                        # genotypes.append(split_header[-1])
                        segments.append(split_header[-2]) # .split("|")[-1])
                        # host_types.append(split_header[-2])
                        species.append(split_first_header[1])
                        # if split_header[4] == "2024-01-01":
                        #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                        # elif split_header[4] == "2025-01-01":
                        #     collection_dates.append("2025")
                        # else: 
                        collection_dates.append(split_header[-1])
                        # collection_dates.append(split_header[-1].split("_")[-1])
                        if num < len(lines): # If we're not at the last line
                            # for i, l in enumerate(lines[num + 1:]):
                            i = num
                            sequence = ""
                            # print(lines[i])
                            # print(lines[i + 1])
                            while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                                sequence = sequence + lines[i + 1].strip()
                                i += 1
                            sequences.append(sequence) # Add next line to sequences
                    except:
                        headers.remove(header)
                        identifiers.remove(identifier)
                        print(header)
                        continue
        f.close()

    
    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["Segment"] = segments
    fasta["Location_Header"] = locations
    # Geo_Location is more complicated
    try:
        fasta["Geo_Location"] = fasta["Location_Header"].apply( lambda x: x.split(" / ")[1] if x.split(" / ") == "Europe" else x.split(" / ")[0])
    except:
        print("Geo Location not found.")
        fasta["Geo_Location"] = fasta["Location_Header"]
    fasta["Date Collected"] = collection_dates
    fasta["Date Collected"] = fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x)
    fasta["Species"] = species
    # fasta["Host_Type"] = host_types
    # fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    # if len(identifiers) == len(fasta):
    fasta["Identifier"] = identifiers
        
    return fasta

In [15]:
# GISAID

os.chdir(downloads_saved) # + "Y2K_swine/")

gisaid_metadata1 = pd.read_excel("gisaid_epiflu_isolates_swine_y2k.xls")

gisaid_fasta1 = fasta_df("gisaid_epiflu_sequence_swine_y2k.fasta")

gisaid_metadata2 = pd.read_excel("gisaid_epiflu_isolates_swine_y2k_2.xls")

gisaid_fasta2 = fasta_df("gisaid_epiflu_sequence_swine_y2k_2.fasta")

gisaid_metadata = pd.concat([gisaid_metadata1, gisaid_metadata2])

gisaid_fasta = pd.concat([gisaid_fasta1, gisaid_fasta2])

# gisaid_metadata
genotypes = list(set(gisaid_metadata["Genotype"].values))

# Separate fastas by segment
fastas, unique_segments = separate_fasta_by_segs(gisaid_metadata, gisaid_fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment
    # print(fastas[0])


EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|PA|2025-07-09
EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|PB2|2025-07-09
EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|PB1|2025-07-09
EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|NP|2025-07-09
EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|NS|2025-07-09
EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|MP|2025-07-09
EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|NA|2025-07-09
EPI_ISL_20333648|2025-wk50-22|A_/_H1N1|HA|2025-07-09
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|NP|2025-05-14
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|NS|2025-05-14
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|MP|2025-05-14
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|NA|2025-05-14
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|HA|2025-05-14
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|PA|2025-05-14
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|PB2|2025-05-14
EPI_ISL_20333649|2025-wk50-23|A_/_H1N2|PB1|2025-05-14
EPI_ISL_20333646|2025-wk50-20|A_/_H1N1|PA|2025-06-04
EPI_ISL_20333646|2025-wk50-20|A_/_H1N1|PB2|2025-06-04
EPI_ISL_20333646|2025-wk50-20|A_/_H1N1|PB

In [16]:
paloma_sequences_gisaid_df = pd.DataFrame()

for fasta in fastas:
    fasta["Isolate"] = fasta["Isolate_Name_x"].apply(lambda x: x.split("/")[-2])
    paloma_sequences_gisaid = paloma_isolates_metadata.merge(fasta, on="Isolate", how="left").drop_duplicates(subset=["Isolate", "Segment"], keep="last")
    paloma_sequences_gisaid_df = pd.concat([paloma_sequences_gisaid_df, paloma_sequences_gisaid])

print(paloma_sequences_gisaid_df)

paloma_sequences_gisaid = paloma_sequences_gisaid_df.dropna(subset="Sequence").sort_values(by="Segment") # .drop_duplicates(subset="strain_name", keep="first")

paloma_sequences_gisaid["full_header"] = paloma_sequences_gisaid["full_header"].apply(lambda x: x + "|" + paloma_sequences_gisaid[paloma_sequences_gisaid["full_header"] == x]["Breed"].values[0]).apply(lambda x: x.split(" ")[0])

print(paloma_sequences_gisaid.columns)

print(paloma_sequences_genbank)

                             strain_name Collection date    Province  \
1             A/swine/Spain/40250-2/2016      11/18/2016      Toledo   
2             A/swine/Spain/40192-1/2016      12/20/2016     Segovia   
3             A/swine/Spain/40250-1/2016      12/20/2016     Segovia   
4             A/swine/Spain/45690-1/2016      12/27/2016      Toledo   
5             A/swine/Spain/45700-2/2016      11/18/2016      Toledo   
..                                   ...             ...         ...   
112           A/swine/Spain/45790-1/2022       9/15/2022      Toledo   
113           A/swine/Spain/36520-1/2022       9/21/2022  Pontevedra   
114           A/swine/Spain/50700-1/2022       10/4/2022    Zaragoza   
115  A/Catalonia/NSAV198309324/2024/H1N1     2024-10-01    Catalonia   
117                  A/Navarra/4050/2022      10/17/2022     Navarra   

         Breed Subtype        Isolate  \
1    White pig     NaN        40250-2   
2    White pig     NaN        40192-1   
3    White p

### De-duplication

In [17]:
paloma_sequences_gisaid = paloma_sequences_gisaid.rename(columns={"Sequence":"sequence"})

# .drop_duplicates(subset=["strain_name", "Segment"],  keep="first")
paloma_sequences = pd.concat([paloma_sequences_genbank, paloma_sequences_gisaid]).drop_duplicates(subset=["Isolate", "Segment"],  keep="first").dropna(subset="full_header").reset_index(drop=True)

print(paloma_sequences[["full_header"]])
print(paloma_sequences.columns)

df_to_fasta(paloma_sequences, "paloma_sequences.fasta", complete_files + "deduplicated/")

                                           full_header
0    >PQ047761.1|A/swine/Spain/6370-1/2018|H1N2|Spa...
1    >PQ047762.1|A/swine/Spain/6370-1/2018|H1N2|Spa...
2    >PQ047763.1|A/swine/Spain/6370-1/2018|H1N2|Spa...
3    >PQ047764.1|A/swine/Spain/6370-1/2018|H1N2|Spa...
4    >PQ047765.1|A/swine/Spain/6370-1/2018|H1N2|Spa...
..                                                 ...
743  >EPI_ISL_18922506|A/swine/Spain/50070-1/2020|H...
744  >EPI_ISL_18921581|A/swine/Spain/31310-1/2020|H...
745  >EPI_ISL_6781281|A/swine/Spain/46314-2/2020|H3...
746  >EPI_ISL_9594287|A/swine/Spain/45690-3/2018|H1...
747  >EPI_ISL_19459695|A/Catalonia/NSAV198309324/20...

[748 rows x 1 columns]
Index(['Accession', 'GenBank_Title', 'Host', 'Collection_Date',
       'SRA_Accession', 'Assembly', 'Isolate', 'Genotype', 'Geo_Location',
       'full_header', 'sequence', 'Segment', 'strain_name', 'Breed',
       'Province', 'genbank_isolate', 'Host_Type', 'Years',
       'Geo_Location_Abrv', 'Host_Type_Breed', '

In [18]:
# De-duplicate from GenBank

gisaid_fastas = []
# for fasta in fastas:
#     # print(fasta)
for serotype in serotypes:
    # print(serotype)
    m_g_s = paloma_sequences.dropna(subset="Subtype_x")[paloma_sequences.dropna(subset="Subtype_x")["Subtype_x"].str.contains(serotype)]
    m_g_s["Serotype"] = serotype
    if len(m_g_s) > 0:
        # # Check to see if isolate is already in GenBank
        # print(m_g_s)
        # m_g_s["Isolate"] = m_g_s["Isolate_Id"]
        # metadata_genoflu["Isolate"] = metadata_genoflu["Isolate"]
        # exp_df = metadata_genoflu.merge(m_g_s, on="Isolate", how="right", indicator=True)
        # # print(exp_df)
        # exp_df["New_Name"] = exp_df[exp_df["_merge"] == "right_only"]["New_Name"].apply(lambda x: x.replace("other_mammal", "swine") if "sw" in x.lower() or "scrofa" in x.lower() or "boar" in x.lower() else x)
        # exp_df = exp_df[exp_df["_merge"] == "right_only"]
        # exp_df["New_Name"] = exp_df["New_Name"].apply(lambda x: "|".join(x.split("|")[:-1]))
        # exp_df = exp_df.drop_duplicates(subset="New_Name", keep="first")
        # m_g_s = m_g_s.drop_duplicates(subset="Isolate", keep="first")
        gisaid_fastas.append(m_g_s)



In [19]:
# print(gisaid_fastas[0].columns)

### Create FASTA Files

In [20]:
# Create FASTA files

os.chdir(complete_files)


for fasta in gisaid_fastas:
    print(fasta)
    for serotype in serotypes:
        if serotype == fasta["Serotype"].values[0]:
            fasta["Segment_Name"] = fasta["Segment"]
            print(list(set(fasta["Segment_Name"].values)))
            for segment in list(set(fasta["Segment_Name"].values)):
                fasta_seg = fasta[fasta["Segment_Name"] == segment]
                fasta_seg["isolate"] = fasta_seg["full_header"].apply(lambda x: x.split("/")[3] if "human" not in x else x.split("/")[2])
                fasta_seg = fasta_seg.drop_duplicates(subset="isolate", keep="first")
                fasta_seg.to_csv("GISAID_" + str(fasta_seg["Segment_Name"].values[0]) + "_" + str(serotype) + "_metadata_Paloma.csv")
                fasta_seg = fasta_seg.reset_index()
                # if len(fasta["Genotype_y"].values) > 0: # If we have assigned genotypes, including "Unassigned"
                file_name = "GISAID_" + str(fasta_seg["Segment_Name"].values[0]) + "_" + str(serotype) + "_Paloma.fasta"
                output_file = open(complete_files + file_name, "w")

                for index, row in fasta_seg.iterrows():
                    name = fasta_seg.loc[index, "full_header"]
                    name = name.replace(" ", "_")
                    # names.append(name)
                    sequence = fasta_seg.loc[index, "sequence"]
                    # First is header, second is sequence
                    output_file.write(name + "\n")
                    output_file.write(sequence + "\n")
                
                output_file.close()

    Accession GenBank_Title        Host Collection_Date SRA_Accession  \
348       NaN           NaN       Human      2022-10-17           NaN   
351       NaN           NaN       Swine      2020-05-19           NaN   
352       NaN           NaN       Swine      2020-05-22           NaN   
353       NaN           NaN       Swine      2020-05-26           NaN   
354       NaN           NaN       Swine      2020-06-15           NaN   
..        ...           ...         ...             ...           ...   
742       NaN           NaN       Swine      2020-05-26           NaN   
743       NaN           NaN       Swine      2020-05-22           NaN   
744       NaN           NaN       Swine      2020-05-19           NaN   
746       NaN           NaN  Sus scrofa      2018-01-31           NaN   
747       NaN           NaN       Human      2024-10-01           NaN   

    Assembly        Isolate      Genotype Geo_Location  \
348      NaN           4050  Not assigned      Navarra   
351    

### Concatenation

In [21]:
# Now concatenate GISAID and GenBank

genbank_files = []
gisaid_files = []
for dirpath, dirs, files in os.walk(complete_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if "GenBank" in file_name and ".fasta" in file_name:
            genbank_files.append(file_name)
        elif "GISAID" in file_name and ".fasta" in file_name:
            gisaid_files.append(file_name)
        else:
            continue
    break 

for file1 in genbank_files:
    if len(file1.split("_")) == 4:
        partial_file_name = "_".join(file1.split("_")[-3:])
    else:
        partial_file_name = "_".join(file1.split("_")[-3:])
    for file2 in gisaid_files:
        if len(file2.split("_")) == 4:
            partial_file_name_gisaid = "_".join(file2.split("_")[-3:])
        else:
            partial_file_name_gisaid = "_".join(file2.split("_")[-3:])
        if partial_file_name == partial_file_name_gisaid:
            with open(complete_files + partial_file_name, 'w') as outfile:
                with open(file1) as f1:
                    for line in f1:
                        outfile.write(line)
                    f1.close()
                with open(file2) as f2:
                    for line in f2:
                        outfile.write(line)
                    f2.close()
                outfile.close()

## Second De-Duplication

In [22]:
output_path = complete_files + "deduplicated/"

if not os.path.exists(output_path): # checking if the directory exists or not
    os.makedirs(output_path) # if the directory is not present then create it

for dirpath, dirs, files in os.walk(complete_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name and "GISAID" not in file_name and "GenBank" not in file_name and len(file_name.split("/")[-1].split("_")) == 3:
        # if ".fasta" in file_name and "GenBank" in file_name:
            df = df_from_fasta(file_name)
            # print(df["full_header"])
            print(len(df["full_header"]))
            df["isolate"] = df["full_header"].apply(lambda x: x.split("/")[3] if "human" not in x else x.split("/")[2])
            df = df.drop_duplicates(subset="isolate", keep="first")
            df = df[df["full_header"].str.contains(".*\\|20.*\\|.*")] # Make sure everything is 2000+
            print(len(df["full_header"]))
            df_to_fasta(df, file_name.split("/")[-1], complete_files)
    break



85
67
9
7
37
28
57
46
85
67
9
7
37
28
57
46
85
67
9
7
37
28
57
46
85
67
9
7
37
28
57
46
85
67
9
7
37
28
57
46
83
65
9
7
36
27
56
45
84
66
9
7
37
28
56
45
84
66
9
7
36
27
57
46


In [23]:
# Modify files so only HA and NA are split
segments_seen = []
for dirpath, dirs, files in os.walk(complete_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        segment = file_name.split("/")[-1].split("_")[0]
        # print(segment)
        if segment != "NA" and segment != "HA" and "rejects" not in file_name and "paloma_sequences" not in file_name and "GISAID" not in file_name and "GenBank" not in file_name and segment not in segments_seen:
            file_list = []
            # Merge the segments
            for file2 in files:
                file_name2 = os.path.join(dirpath, file2)
                segment2 = file_name2.split("/")[-1].split("_")[0]
                if segment == segment2:
                    # Concatenate files
                    file_list.append(file_name2)
            df_list = []
            for listed_file in file_list:
                df = df_from_fasta(listed_file)
                df["isolate"] = df["full_header"].apply(lambda x: x.split("/")[3] if "human" not in x else x.split("/")[2])
                df = df.drop_duplicates(subset="isolate", keep="first")
                df_list.append(df)
            concatenated_df = pd.concat(df_list).reset_index(drop=True).drop_duplicates(subset="full_header", keep="first")
            # print(concatenated_df)
            df_to_fasta(concatenated_df, segment + "_Paloma.fasta", output_path)
            segments_seen.append(segment)
            print(segments_seen)
        elif segment == "NA" or segment == "HA" and "rejects" not in file_name and "paloma_sequences" not in file_name and "GISAID" not in file_name and "GenBank" not in file_name and segment not in segments_seen:
            df = df_from_fasta(file_name)
            df["isolate"] = df["full_header"].apply(lambda x: x.split("/")[3] if "human" not in x else x.split("/")[2])
            df = df.drop_duplicates(subset="isolate", keep="first").drop_duplicates(subset="full_header", keep="first")
            df_to_fasta(df, "_".join(file_name.split("/")[-1].split("_")[:2]) + "_Paloma.fasta", output_path)
        else:
            continue
    break 

['MP']
['MP', 'NP']
['MP', 'NP', 'NS']
['MP', 'NP', 'NS', 'PA']
['MP', 'NP', 'NS', 'PA', 'PB1']
['MP', 'NP', 'NS', 'PA', 'PB1', 'PB2']


## GISAID (again)

In [24]:
# downloads_saved = home + "Other/Paloma/Y2K_swine/" 

# complete_files = downloads_saved + "complete/" 
# if not os.path.exists(complete_files): # checking if the directory exists or not
#     os.makedirs(complete_files) # if the directory is not present then create it



In [25]:
# all_metadata_files = []
# all_fasta_files = []

# # If we have multiple files, name them nicely
# for dirpath, dirs, files in os.walk(downloads_saved):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         # file_name = "_".join(file_name.split(" "))
        
#         os.rename(file_name, "_".join(file_name.split(" ")).replace("(", "").replace(")", ""))

#         print(file_name)

#         # Now go through files and get contents
#         if ".xls" in file_name:
#             metadata = pd.read_excel(file_name)
#             all_metadata_files.append(metadata)
#         # if ".csv" in file_name:
#         #     metadata = pd.read_csv(file_name)
#         #     all_metadata_files.append(metadata)
#         if ".fasta" in file_name:
#             fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
#             # print(fasta_file[fasta_file["Geo_Location"] != "USA"])
#             # break 
#             all_fasta_files.append(fasta_file)
#     break 

In [26]:
# # Separate FASTA files into 8 different files based on segment

# def separate_fasta_by_seg(metadata, fasta, animals_df): 

#     fasta = fix_animals(fasta, animals_df)
#     # Dummy host type -- we'll actually add this in later
#     # b313_fasta["Host_Type"] = "other"
#     # d11_fasta["Host_Type"] = "other"

#     unique_segments = list(set(fasta["Segment"]))
#     # genotypes = ["B3.13", "D1.1"]
#     # genotype_fastas = {"B3.13": b313_fasta, "D1.1": d11_fasta}

#     # “>EPI_ID/Isolate_name|subtype|collection_date|host_type|genotype”

#     segment_fastas = []
#     for seg in unique_segments:

#         xls = metadata

#         # print(xls)
#         # print("XLS: ", metadata["Genotype"])

#         # print(d11_xls)

#         # FASTA
#         # if "Identifier" in fasta.columns:
#         #     mask = fasta["Identifier"].isin(xls['Isolate_Id'])
#         # else:           
#         #     mask = fasta['Isolate_Id'].isin(xls['Isolate_Id'])
#         #     fasta["Identifier"] = fasta["Isolate_Id"]

#         # fasta_seg_pre = fasta[mask]
#         # print(fasta_seg_pre)
#         xls["Identifier"] = xls["Isolate_Id"]
#         fasta_seg_pre = fasta.merge(xls, on="Identifier")

#         # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
#         fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]
#         print(fasta_seg)

#         # Rename sequences 
#         new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name_x"] + "|" + fasta_seg["Subtype_x"].apply(lambda x: x.split("_")[-1]) + "|" + fasta_seg["Location_y"].apply(lambda x: x.split("/")[1].replace(" ", "")) + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|swine\n"
#         fasta_seg["New_Name"] = new_name
#         # print(fasta_seg["New_Name"])

#         segment_fastas.append(fasta_seg)
#         # print(fasta_seg)

#     return segment_fastas, unique_segments

# # Separate fastas by segment
# segment_fastas = []
# unique_animals_all = []
# for i, fasta in enumerate(all_fasta_files):
#     # print(all_fasta_files)
#     # print(fasta)
#     # print(i)
#     metadata = all_metadata_files[i]
#     # print(metadata["Clade"])
#     # break 
#     # print(fasta.loc[i, "Isolate_Name"])
    
#     unique_animals = sort_animals(fasta) # Find unique animals
#     # print("Animals: ", unique_animals)
#     unique_animals_all.append(unique_animals)

#     os.chdir(references)
#     animals_ref = pd.read_csv("animals_ref.csv")
#     print(fasta)
#     fastas, unique_segments = separate_fasta_by_seg(metadata, fasta, animals_ref) # Separate the fasta dataframes into 8 different files based on segment
#     # print(fastas[0])

#     for fasta in fastas:
#         segment_fastas.append(fasta)

In [27]:
# # Find animals to sort, if needed

# os.chdir(references)

# # Flatten unique_animals
# every_unique_animal = []
# for l in unique_animals_all:
#     for animal in l:
#         every_unique_animal.append(animal)

# # Rename host type

# unique_animals_set = list(set(every_unique_animal))
# animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "other"])
# animals_df["other"] = unique_animals_set # to sort

# # If animal not in ref1, put in ref2

# common_animals = []
# # Check if animals in unique_animals_set are in ref1
# for animal in unique_animals_set:
#     for col in animals_ref.columns:
#         if animal in animals_ref[col].values and type(animal) == str:
#             common_animals.append(animal)

# print(common_animals)
# print(len(common_animals))

# different_animals = []
# for animal in unique_animals_set:
#     if animal not in common_animals:
#         different_animals.append(animal)

# print(different_animals)

# print(animals_ref)

# # Add to dataframe
# animals_df = animals_ref
# # Make different_animals same length as dataframe, if shorter
# if len(different_animals) < len(animals_df):
#     number_of_times_to_add_nan = len(animals_df) - len(different_animals)
#     for i in range(number_of_times_to_add_nan):
#         different_animals.append(float('nan'))
# # If longer
# else:
#     number_of_times_to_add_nan = len(different_animals) - len(animals_df)
#     nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
#     for i in range(number_of_times_to_add_nan):
#         animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

# animals_df["new"] = (different_animals)

# print(animals_df)

# os.chdir(references)
# animals_df.to_csv("animals_ref_to_sort.csv")

In [28]:
# input("Check animal output.")

In [29]:
# gisaid_fastas = []
# for fasta in segment_fastas:
#     if fasta["Segment"].values[0] == "HA":
#         print("HA")
#         for s in ["H1", "H3"]:
#             m_g_s = fasta[fasta["Subtype_x"].str.contains(s)]
#             m_g_s["Serotype"] = s
#             if len(m_g_s) > 0:
#                 print(m_g_s["Subtype_x"])
#                 print(m_g_s)
#                 # m_g_s["New_Name"] = m_g_s["Header"] # .apply(lambda x: x.replace("other_mammal", "swine") if "other_mammal" in x else x)
#                 gisaid_fastas.append(m_g_s)
#     elif fasta["Segment"].values[0] == "NA":
#         print("NA")
#         for se in ["N1", "N2"]:
#             m_g_s = fasta[fasta["Subtype_x"].str.contains(se)]
#             m_g_s["Serotype"] = se
#             if len(m_g_s) > 0:
#                 # print(exp_df)
#                 # m_g_s["New_Name"] = m_g_s["Header"] # .apply(lambda x: x.replace("other_mammal", "swine") if "other_mammal" in x else x)
#                 gisaid_fastas.append(m_g_s)
#     else:
#         '''all other segments'''
#         if len(fasta) > 0:
#             fasta["Serotype"] = "None"
#             # fasta["New_Name"] = fasta["Header"] # .apply(lambda x: x.replace("other_mammal", "swine") if "other_mammal" in x else x)
#             gisaid_fastas.append(fasta)

# print(gisaid_fastas[0]["New_Name"].values[0].split("|")[3:])

In [30]:
# for gisaid_fasta in gisaid_fastas:
#     print(gisaid_fasta)
#     if "Segment_x" in gisaid_fasta.columns:
#         gisaid_fasta["Segment"] = gisaid_fasta["Segment_x"]
#     if "Isolate_Id_x" in gisaid_fasta.columns:
#         gisaid_fasta["Isolate"] = gisaid_fasta["Isolate_Id_x"]
#     print(gisaid_fasta["Segment"])

# # Remove Paloma's sequences
# os.chdir(downloads_saved)
# paloma_isolates = pd.read_csv("paloma_isolates.csv")

# for gisaid_fasta in gisaid_fastas:
#     for isolate in paloma_isolates["isolate"].values:
#         print(isolate)
#         if isolate in gisaid_fasta["Isolate"]:
#             gisaid_fasta = gisaid_fasta[gisaid_fasta["Isolate"] != isolate]

#     # Remove duplicates
#     gisaid_fasta = gisaid_fasta.drop_duplicates(subset="Header", keep="first")


In [31]:
# # Create FASTA files

# os.chdir(complete_files)

# for df in gisaid_fastas:
#     for serotype in serotypes:
#         if serotype == df["Serotype"].values[0]:
#             print(serotype)
#             if "Segment_y" in df.columns:
#                 df["Segment_Name"] = df["Segment_y"]
#             else:
#                 df["Segment_Name"] = df["Segment"]
#             # print(df.columns)
#             df.to_csv("GISAID_" + df["Segment_Name"].values[0] + "_" + serotype + "_metadata_Paloma.csv")
#             # df = df.reset_index()
#             # if len(df["Genotype_y"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
#             file_name = "GISAID_" + df["Segment_Name"].values[0] + "_" + serotype + "_Paloma.fasta"
#             output_file = open(complete_files + file_name, "a+")

#             for index, row in df.iterrows():
#                 name = df.loc[index, "New_Name"]
#                 name = name.replace(" ", "_")
#                 # names.append(name)
#                 sequence = df.loc[index, "Sequence"]
#                 # First is header, second is sequence
#                 output_file.write(name)
#                 output_file.write(sequence + "\n")
#     if df["Serotype"].values[0] == "None":
#         # df["Segment_Name"] = df["Segment_x"]
#         if "Segment" in df.columns:
#             df["Segment_Name"] = df["Segment"]
#         else:
#             df["Segment_Name"] = ""
#         # print(df)
#         df.to_csv("GISAID_" + df["Segment_Name"].values[0] + "_metadata_Paloma.csv")
#         # df = df.reset_index()
#         # if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
#         file_name = "GISAID_" + df["Segment_Name"].values[0] + "_Paloma.fasta"
#         output_file = open(complete_files + file_name, "a+")

#         for index, row in df.iterrows():
#             name = df.loc[index, "New_Name"]
#             name = name.replace(" ", "_")
#             # names.append(name)
#             sequence = df.loc[index, "Sequence"]
#             # First is header, second is sequence
#             output_file.write(name)
#             output_file.write(sequence + "\n")
#         output_file.close()